## Builtin Tools in langchain

### 1. DuckDuckGo search tool

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

#Here tools are also runnables, so they have their own invoke function
results = search_tool.invoke('AI news')

print(results)

AI News is an umbrella term for the continuously updated informational ecosystem reporting on artificial intelligence developments, including research results, model releases, product launches, corporate strategy, regulation, labor impacts, security… 2 days ago - AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide. 8 hours ago - News coverage on artificial intelligence and machine learning tech, the companies building them, and the ethical issues AI raises today. 2 days ago - Artificial Intelligence: Read latest updates on AI like Google AI, ChatGPT, Google Lamda, Bard chatbot and more along with latest news as AI technology advances and makes new progress. All get detailed articles on AI related queries like what is AI, types of artificial intelligence, its applications and future. 12 hours ago - An overview of Google’s new global partnerships and funding announcements at the AI Impact Summit in In

### 2. Shell Tool

In [ ]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('whoami')

print(results)

Executing command:
 whoami
dragon



/home/dragon/.virtualenvs/langchain-campusX/lib/python3.12/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


In [7]:
results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
Tools-in-langchain.ipynb



## Custom Tools in langchain

### Method 1- Using tool decorartor

In [8]:
from langchain_core.tools import tool

#### Step 1: Create a function

Here docstring is very important because LLM will understand the purpose of this function only with the help of this doc-string

In [9]:
def multiply(a,b):
    """ Multiply two numbers """
    return a*b 

#### Step 2: Add Type Hints

Here, while defining the function, we will drop the hints so that LLM will know which data type to send, this is not enforced in python , but still its a suggestion

In [10]:
def multiply(a: int, b: int) -> int:
    """ Multiply two numbers """
    return a*b 

#### Step 3: Add tool decorator

Now, this decorator is what makes this function accessible to the LLM, so the entire magic lies in this decorator only

In [28]:
@tool
def multiply(a: int, b: int)-> int:
    """ Multiply two numbers """
    return a*b 

#### Now, use the tool


Now, this tool is also a runnable in langchain, so we need to use invoke function to get the output.


In [13]:
result = multiply.invoke(
    {"a": 2, "b": 3}
)

print(result)

6


#### Lets explore the properties of this tool


Now, every tool has these 3 attributes mentioned below.

In [14]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


This is also true for inbuilt tools

In [15]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


#### Now, when we send this tool to an LLM, lets look at what it actually sees

In [16]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers ', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


### Method 2 - Using Structured Tool

First define the pydantic data-class and the function which we want to use as tool

In [17]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(required= True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

def multiply_func(a: int, b: int) -> int:
    return a*b

/tmp/ipykernel_169032/1418027348.py:5: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required= True, description="The first number to add")
/tmp/ipykernel_169032/1418027348.py:6: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


Now, we define the tool, by combining them both
1. The function is passed
2. The pydantic class defined above will be passed as the arguments schema 

In [18]:
multiply_tool = StructuredTool.from_function(
    func = multiply_func, # The function is passed here
    name = "multiply",
    description= "Multiply two numbers",
    args_schema=MultiplyInput # The pydantic class which we defined above is passed as the arguments schema
)

Now, we will call the tool

In [21]:
result = multiply_tool.invoke({"a": 3, "b" : 4})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

12
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


Now, lets check if it really enforces the constraints or not by passing string values instead of integers.

There are 2 scenarios here:
1. If the passed string can be converted to an integer, then it will do it.
2. If it cannot be converted to an integer, it will throw an error

In [24]:
result = multiply_tool.invoke({"a": "3", "b" : 4})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

12
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


Now, above we passed a number in the form of a string , it worked, but now lets pass "-" and see if it throws an error

In [25]:
result = multiply_tool.invoke({"a": "-", "b" : 4})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

ValidationError: 1 validation error for MultiplyInput
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='-', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

Now, lets test the same thing on the tool defined with @tool decorator

So, apparently @tool is also updated to enforce the rules just like pydantic, its not like previous langchain versions

In [26]:
result = multiply.invoke({"a": "-", "b" : 4})
print(result)
print(multiply.name)
print(multiply.description)
print(multiply.args)

ValidationError: 1 validation error for multiply
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='-', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

### Method 3 - Using BaseTool Class

The Speciality of this method is , we get more control over the tool like making it async also while defining it, which we will not get in the above 2 methods.

In [32]:
from langchain.tools import BaseTool
from typing import Type 
from pydantic import BaseModel

Define the argument schema first with the help of pydantic

In [33]:
class MultiplyInput(BaseModel):
    a: int = Field(required= True, description="The first number to add")
    b: int = Field(required=True, description="The second number to add")

/tmp/ipykernel_169032/563945432.py:2: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  a: int = Field(required= True, description="The first number to add")
/tmp/ipykernel_169032/563945432.py:3: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  b: int = Field(required=True, description="The second number to add")


Now define the tool by defining a class which inherits this BaseTool class

It has 3 attributes and 1 function:
1. 2 attributes are "name" and "description" which are type string
2. Third attribute is the argument schema which will be of type pydantic BaseClass which itself is a dataclass.
3. The function should be named _run() thats how the BaseTool() class is defined, so that whenever a tool is called , it will automatically call the _run() function

In [ ]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a*b 

Lets initialize and call the tool

In [37]:
multiply_tool = MultiplyTool()

result = multiply_tool.invoke({'a':5, "b": 10})
print(result)

50


## ToolKits in langchain

This is just a way of organising the tools which are similar or etc in to a group for convinient usage

First lets define the tools which are related to something lets say mathematical operations

In [38]:
from langchain_core.tools import tool

#custom tools
@tool
def add(a:int, b:int) -> int:
    """Add two numbers"""
    return a+b

@tool
def multiply(a:int, b:int)->int:
    """Multiply two numbers"""
    return a*b


Now, lets define the toolkit in the form of a class

In [39]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

Lets initialize the toolkit and see how this works

In [40]:
toolkit = MathToolkit()

#Now the tools will be in the form of a list so lets get that list
tools = toolkit.get_tools()

#iterate through the list and the print the name and descriptions
for tool in tools:
    print(tool.name, "==>", tool.description)

add ==> Add two numbers
multiply ==> Multiply two numbers


## Tool binding (Giving LLM access to the tools)

In [41]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

Lets create tool first


In [42]:
@tool
def multiply(a: int, b: int) -> int:
    """Given 2 numbers a and b, this tool returns their product"""
    return a*b

#lets test this by invoking it
print(multiply.invoke({"a": 3, "b":2}))

6


Now, tool binding

NOTE: not every llm that we have , we can bind the tools, the LLMs which have the tool calling capability will only have this option

In [44]:
llm = ChatOpenAI()

#Simply call this function
llm_with_tools = llm.bind_tools([multiply])

## Tool calling

Now, lets test whether it generates a tool call or not etc

In [45]:
llm_with_tools.invoke("Hi, how are you?")

AIMessage(content="Hello! I'm here and ready to help. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 61, 'total_tokens': 79, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DdyxdKkHAX9dXMUG5UyHp87v92lGS', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e122e-77c8-7cf2-88aa-920d41afceca-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 61, 'output_tokens': 18, 'total_tokens': 79, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

The above , we see that it didnt felt to the need to call the tool, so it responded normally, now lets test with actual question which will trigger this tool call.

In [46]:
llm_with_tools.invoke('can you multiply 3 iwth 10?')

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 65, 'total_tokens': 82, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-De0BmkNpBdwzggQaa9zAu2uyuJFRQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e1276-7e23-7763-8bd1-bd6914b87f95-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_NfSfD9dOtdYIIylEgqwmCIYH', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 65, 'output_tokens': 17, 'total_tokens': 82, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Here, above we see that content is empty, also there is a tool_calls with the arguments mentioned above.

Since, its an attribute, we can directly fetch it. It is in the form is a list to facilitate more than 1 tool, but since we have a single tool, there will only be one call in a list

In [47]:
result = llm_with_tools.invoke('can you multiply 3 iwth 10?')
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 10},
  'id': 'call_8oLYrWuwD3DLTZEq43wBrQjz',
  'type': 'tool_call'}]

now, we need to execute the tool or function

In [48]:
multiply.invoke({'a': 3, 'b': 10})

30

we can also do it like this, we can send the entire tool call dictionary.

In [49]:
multiply.invoke(result.tool_calls[0])

ToolMessage(content='30', name='multiply', tool_call_id='call_8oLYrWuwD3DLTZEq43wBrQjz')

Here, we see one more message type, other than Human message, System message and AI message etc, which is tool message, which appears when ever a tool call in executed.

Now, the speciality of this tool call is that, we can send this whole message object directly to the LLM to get the final output.

#### Lets make this a little systematic like make all the messages in to message objects instead of plain texts


Lets also maintain all the messages in a list like a chat history kind off

In [50]:
query = HumanMessage('Can you multiply 3 with 9?')

messages = [query]

messages

[HumanMessage(content='Can you multiply 3 with 9?', additional_kwargs={}, response_metadata={})]

In [52]:
result = llm_with_tools.invoke(query.content)

result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 64, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-De0vdenWhx7jCOXOavl5RC7UckgTF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e12a1-e0e8-7b21-b791-8f8c79ef8105-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 9}, 'id': 'call_vYwqKzAzInDA901KNgLyyspS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 17, 'total_tokens': 81, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

now lets add this Ai message also to the messages list

In [55]:
messages.append(result)

messages

[HumanMessage(content='Can you multiply 3 with 9?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 64, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-De0vdenWhx7jCOXOavl5RC7UckgTF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e12a1-e0e8-7b21-b791-8f8c79ef8105-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 9}, 'id': 'call_vYwqKzAzInDA901KNgLyyspS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 17, 'total_tokens': 81, 'input_token_details': {'audio': 0, 'cache_re

In [56]:
tool_result = multiply.invoke(result.tool_calls[0])

tool_result

ToolMessage(content='27', name='multiply', tool_call_id='call_vYwqKzAzInDA901KNgLyyspS')

Lets add this tool message also to the messages list

In [57]:
messages.append(tool_result)

messages

[HumanMessage(content='Can you multiply 3 with 9?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 64, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-De0vdenWhx7jCOXOavl5RC7UckgTF', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e12a1-e0e8-7b21-b791-8f8c79ef8105-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 9}, 'id': 'call_vYwqKzAzInDA901KNgLyyspS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 64, 'output_tokens': 17, 'total_tokens': 81, 'input_token_details': {'audio': 0, 'cache_re

#### Now, that we have this history type list, we will pass this entire thing to LLM  and see what happens

In [58]:
llm_with_tools.invoke(messages).content

'The product of 3 and 9 is 27.'

## Currency conversion tool example

1st lets create tools:

There will be 2 tools:
1. One which will hit this exchange rate API and get the real time conversion factor between the currencies
2. Second tool which will simply multiply this conversion factor and then gives us the results

Lets define the 1st tool

In [69]:
from langchain_core.tools import tool
from dotenv import load_dotenv
import os

load_dotenv()

exchange_api_key = os.getenv("EXCHANGE_RATE_API")

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """
    This function fetches the currency conversion factor between a given base currency and target currency
    """

    url = f"https://v6.exchangerate-api.com/v6/{exchange_api_key}/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()

Lets test it once by invoking it

In [70]:
get_conversion_factor.invoke({"base_currency": "USD", "target_currency": "INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1778371201,
 'time_last_update_utc': 'Sun, 10 May 2026 00:00:01 +0000',
 'time_next_update_unix': 1778457601,
 'time_next_update_utc': 'Mon, 11 May 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.4767}

lets define the second tool

In [71]:
@tool
def convert(base_currency_value: int, conversion_rate: float) -> float:
    """
    Given a currency conversion rate, this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate  

Lets call and test it

In [72]:
convert.invoke({"base_currency_value": 10, 'conversion_rate': 85.16})

851.5999999999999

Now, lets bind the tools

In [73]:
llm = ChatOpenAI()

llm_with_tools= llm.bind_tools([convert, get_conversion_factor])

Lets define the messages now

In [ ]:
messages = [HumanMessage("what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?")]
messages

[HumanMessage(content='what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={})]

In [79]:
ai_message = llm_with_tools.invoke(messages)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_HPsNQMsEn85eJv6o7KKLbb6J',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_rate': 73.5},
  'id': 'call_PN6hIS9p7jWJByJzUinwO1HE',
  'type': 'tool_call'}]

Here, for the second tool call, it hallucinated the conversion rate instead of taking from the 1st tool, call.
Because we asked both the questions at the same time, it felt the need to call 2 tools, but it does not have the intelligence to take the output of 1 tool and use it as an input to another tool etc.

To solve this we have something in langchain called "Injected tools argument"

Now, while defining the 2nd tool, we need to mention the argument datatype of the 2nd tool to be this injected tool argument data type.
Here, this datatype ,means that we are telling the LLM, not to insert this argument while calling the tool, its the job the developer to insert the value over there while calling the 2nd tool.

In [89]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Given a currency conversion rate, this function calculates the target currency value from a given base currency value
    """

    return base_currency_value * conversion_rate  

Now, lets do this again

In [87]:
llm = ChatOpenAI()

llm_with_tools= llm.bind_tools([convert, get_conversion_factor])

messages = [HumanMessage("what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?")]
messages

[HumanMessage(content='what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={})]

Now, we can see that while calling 'convert' it didnt insert the 2nd argument

In [88]:
ai_message = llm_with_tools.invoke(messages)
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_uUH3qkG737Bo138Rgg6iOJLf',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_YOlOxJ1MQg0CuEBleQerlAdw',
  'type': 'tool_call'}]

In [90]:
messages.append(ai_message)
messages

[HumanMessage(content='what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-De260tkXPunUaOqwYIy7CbbjBxdWl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e12e6-5ad1-7d42-8ea6-caa870a4778c-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_uUH3qkG737Bo138Rgg6iOJLf', 'type': 'tool_call'}, {'name': 'convert', 'arg

Now, that we got the tool calls json, we just need to invoke the tools , while invoking we can insert the 2nd argument for the 2nd tool, because we have reserved that to do this.

In [91]:
import json

for tool_call in ai_message.tool_calls:
    
    #Lets execute the 1st tool and get the value of the conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1=get_conversion_factor.invoke(tool_call)
        print(tool_message1)
        #fetch this conversion rate from this tool_message1
        #here the content is the json , but in the form of a string,
        #so we need to convert it in to actual json and then fetch the keys
        conversion_rate= json.loads(tool_message1.content)['conversion_rate']
        print(conversion_rate)
        #Append this tool message to messages list
        messages.append(tool_message1)
    #execute the 2nd tool using the conversion rate from the 1st tool
    if tool_call['name'] == 'convert':
        #First we will fetch the current argument
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2 = convert.invoke(tool_call)
        #append this tool messsage also to the messages list
        messages.append(tool_message2)

content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1778371201, "time_last_update_utc": "Sun, 10 May 2026 00:00:01 +0000", "time_next_update_unix": 1778457601, "time_next_update_utc": "Mon, 11 May 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 94.4767}' name='get_conversion_factor' tool_call_id='call_uUH3qkG737Bo138Rgg6iOJLf'
94.4767


In [92]:
messages

[HumanMessage(content='what is the conversion factor between USD and INR, and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 123, 'total_tokens': 175, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-De260tkXPunUaOqwYIy7CbbjBxdWl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e12e6-5ad1-7d42-8ea6-caa870a4778c-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'call_uUH3qkG737Bo138Rgg6iOJLf', 'type': 'tool_call'}, {'name': 'convert', 'arg

Now, that we have got all the messages history, we will pass this to the LLM to get the final answer

In [94]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 94.4767. \nTherefore, 10 USD is equal to 944.77 INR.'